# 📈 Credit Risk Platform: Exploratory Data Analysis (EDA)
This notebook covers the exploratory analysis of client demographics, credit attributes, and historical payment behaviors to identify signals predictive of credit default risk.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure source modules are importable from notebooks directory
sys.path.append(os.path.abspath(".."))

from src.data.loader import load_dataset
from src.utils.logger import setup_logger

logger = setup_logger("eda_notebook")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Data
We will load the primary training application table containing historical applicants' data.

In [ ]:
data_path = "../data/application_train.csv"
if not os.path.exists(data_path):
    logger.warning(f"Dataset not found at {data_path}. Creating a dummy dataframe for demonstration.")
    # Generate dummy data for illustration
    np.random.seed(42)
    n_samples = 1000
    df = pd.DataFrame({
        'SK_ID_CURR': range(100000, 100000 + n_samples),
        'TARGET': np.random.choice([0, 1], size=n_samples, p=[0.92, 0.08]),
        'NAME_CONTRACT_TYPE': np.random.choice(['Cash loans', 'Revolving loans'], size=n_samples),
        'CODE_GENDER': np.random.choice(['M', 'F'], size=n_samples),
        'FLAG_OWN_CAR': np.random.choice(['Y', 'N'], size=n_samples),
        'CNT_CHILDREN': np.random.poisson(0.5, size=n_samples),
        'AMT_INCOME_TOTAL': np.random.lognormal(11.5, 0.5, size=n_samples),
        'AMT_CREDIT': np.random.exponential(500000, size=n_samples) + 50000,
        'DAYS_BIRTH': -np.random.randint(7300, 25000, size=n_samples),
        'EXT_SOURCE_1': np.random.uniform(0.1, 0.9, size=n_samples),
        'EXT_SOURCE_2': np.random.uniform(0.05, 0.95, size=n_samples),
        'EXT_SOURCE_3': np.random.uniform(0.01, 0.85, size=n_samples)
    })
    # Introduce some missing values
    df.loc[df.sample(frac=0.15).index, 'EXT_SOURCE_1'] = np.nan
    df.loc[df.sample(frac=0.05).index, 'EXT_SOURCE_3'] = np.nan
else:
    df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Target Variable Distribution
Let's look at the class imbalance in credit default (TARGET = 1).

In [ ]:
target_counts = df['TARGET'].value_counts(normalize=True) * 100
print("Target Distribution (%):")
print(target_counts)

plt.figure(figsize=(6, 5))
sns.countplot(x='TARGET', data=df)
plt.title('Distribution of Target Variable (0 = Repaid, 1 = Defaulted)')
plt.ylabel('Count')
plt.show()

## 3. Analysis of External Credit Scores
External sources (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`) are typically the strongest predictors of default. Let's analyze their correlations and distributions.

In [ ]:
cols_scores = ['TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
corr = df[cols_scores].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f', vmin=-1, vmax=1)
plt.title('Correlation Matrix of External Score Ratings')
plt.show()

### Score Distributions by Target Class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']):
    sns.kdeplot(data=df, x=col, hue='TARGET', common_norm=False, fill=True, alpha=0.4, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel('Score')

plt.tight_layout()
plt.show()

## 4. Key Takeaways
1. **Imbalanced Classes**: Extreme class imbalance exists (~92% vs ~8%). Models must account for this (e.g., scale_pos_weight or class weights).
2. **Highly Predictive Score Ratings**: Lower values of EXT_SOURCE features correlate directly with significantly higher default probabilities.